# Fine-tuning PaliGemma-3B for Multi-Turn Visual QA
**Platform:** Kaggle (P100 16 GB GPU)  
**Method:** float16 LoRA — no quantization needed, fits P100 comfortably

**Before running:**
- Kaggle Settings → Accelerator → **GPU P100**
- Kaggle Settings → Internet → **On**
- Add-ons → Secrets → add **HF_TOKEN** = your HuggingFace Classic token

**Run order:** Step 0 → Step 1 → Steps 2–12 top to bottom  
**Training time:** ~2–3 hours. Output saved to `/kaggle/working/dlcv_paligemma/`

## Step 0 — Verify GPU
Run first to confirm T4 is assigned.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — Settings → Accelerator → GPU P100'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA : {torch.version.cuda}')
print(f'Torch: {torch.__version__}')

## Step 1 — Install Dependencies
Runs once. No restart needed on Kaggle — continue straight to Step 2.

In [ ]:
import subprocess, sys

# Upgrade bitsandbytes first — Kaggle pre-installs an old version
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade', 'bitsandbytes'], check=True)

# Core packages — no datasets lib (we download JSON directly via requests)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.44.0',
    'peft==0.12.0',
    'accelerate==0.33.0',
    'Pillow',
    'requests',
    'matplotlib',
    'tqdm',
], check=True)

# ── MUST be the very last pip command ────────────────────────────────────────
# accelerate/peft/transformers pull numpy 1.x as a transitive dep.
# Kaggle's torch + torchvision C-extensions are compiled for numpy 2.x.
# Mixing 1.x runtime with 2.x .so files → ValueError: numpy.dtype size changed.
# --force-reinstall --no-deps forces a clean 2.0.2 installation without
# downgrading any of the packages we just installed above.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps', 'numpy==2.0.2'], check=True)

print('✅ All packages installed — continue to Step 2')

## Step 2 — Imports & Paths

In [ ]:
import sys, os, json, random, requests
from io import BytesIO

# ── Patch triton.ops BEFORE importing PEFT ──────────────────────────────────
# Kaggle's pre-installed bitsandbytes imports triton.ops.matmul_perf_model
# which was removed in triton 2.x. Mock it — not used in float16 LoRA training.
try:
    from triton.ops.matmul_perf_model import early_config_prune  # noqa: test import
except (ImportError, ModuleNotFoundError):
    from unittest.mock import MagicMock
    _mock = MagicMock()
    sys.modules['triton.ops']                   = _mock
    sys.modules['triton.ops.matmul_perf_model'] = _mock
    print('Patched triton.ops (removed in triton 2.x, not needed for float16)')

import numpy as np
print(f'numpy {np.__version__}')

import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from huggingface_hub import login

# Kaggle output paths — files appear in the Output tab after the session ends
SAVE_DIR   = '/kaggle/working/dlcv_paligemma'
OUTPUT_DIR = '/kaggle/working/paligemma_checkpoint'
os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'✅ All imports OK')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Save: {SAVE_DIR}')

## Step 3 — HuggingFace Login

In [ ]:
# ── HuggingFace Login ───────────────────────────────────────────────────────
# Use a CLASSIC token (not fine-grained) — fine-grained tokens block gated repos.
# Add it once: Kaggle → Add-ons → Secrets → Name: HF_TOKEN → Value: hf_...
# Accept PaliGemma license at: huggingface.co/google/paligemma-3b-pt-224

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    HF_TOKEN = ''   # ← paste token here if Secrets not configured

assert HF_TOKEN, (
    'No HF token found!\n'
    'Go to Add-ons → Secrets → add Name=HF_TOKEN, Value=hf_...\n'
    'Or paste your Classic token directly into HF_TOKEN above.'
)

login(token=HF_TOKEN)

from huggingface_hub import model_info
try:
    info = model_info('google/paligemma-3b-pt-224', token=HF_TOKEN)
    print(f'✅ Token OK — PaliGemma access confirmed (gated: {info.gated})')
except Exception as e:
    print(f'❌ Token error: {e}')
    print('→ Use a CLASSIC token and accept the license at:')
    print('  huggingface.co/google/paligemma-3b-pt-224')

## Step 4 — Load & Build Dataset
Uses LLaVA-Instruct-150K conversation data (multi-turn) + COCO images fetched on-demand.
~2,000 samples total. Takes ~3 min to build.

In [ ]:
# Download LLaVA-Instruct-150K JSON directly — no `datasets` library needed.
# Avoids pyarrow's numpy ABI dependency entirely.
print('Downloading LLaVA-Instruct-150K from HuggingFace (~80 MB)...')
_url = ('https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K'
        '/resolve/main/conversation_58k.json')
_r = requests.get(_url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=180)
_r.raise_for_status()
llava_ds = _r.json()   # plain Python list of dicts: {id, image, conversations}
print(f'✅ Loaded {len(llava_ds)} conversation samples')

In [ ]:
NUM_SAMPLES = 2000
EVAL_SPLIT  = 0.1
COCO_URL    = 'http://images.cocodataset.org/train2017/{:012d}.jpg'
IMG_CACHE   = {}   # in-memory image cache

random.seed(42)


def _parse_coco_id(image_id: str) -> int:
    """Handle plain integers AND 'COCO_train2014_000000000009' style IDs."""
    s = str(image_id).strip()
    if '_' in s:
        s = s.split('_')[-1]   # take the numeric suffix
    return int(s)


def fetch_image(image_id: str) -> Image.Image:
    """Download COCO image by ID with caching."""
    if image_id in IMG_CACHE:
        return IMG_CACHE[image_id]
    try:
        url  = COCO_URL.format(_parse_coco_id(image_id))
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        img  = Image.open(BytesIO(resp.content)).convert('RGB')
    except Exception:
        img = Image.new('RGB', (224, 224), (128, 128, 128))
    IMG_CACHE[image_id] = img
    return img


def parse_llava(sample: dict) -> dict | None:
    """Parse LLaVA-Instruct sample into unified format."""
    raw   = sample.get('conversations', [])
    turns = []
    for c in raw:
        role    = 'user' if c['from'] == 'human' else 'assistant'
        content = c['value'].replace('<image>', '').strip()
        if content:
            turns.append({'role': role, 'content': content})
    if len(turns) < 2:
        return None
    return {
        'image_id'     : str(sample.get('id', '0')),
        'conversations': turns,
    }


# Sample & filter
indices = random.sample(range(len(llava_ds)), min(NUM_SAMPLES * 2, len(llava_ds)))
samples = []
for i in indices:
    s = parse_llava(llava_ds[i])
    if s is not None:
        samples.append(s)
    if len(samples) >= NUM_SAMPLES:
        break

random.shuffle(samples)
split      = int(len(samples) * (1 - EVAL_SPLIT))
train_data = samples[:split]
val_data   = samples[split:]

print(f'✅ Train: {len(train_data)} samples')
print(f'✅ Val:   {len(val_data)} samples')
print('\nExample conversation:')
for t in train_data[0]['conversations'][:4]:
    print(f"  [{t['role']:9s}]: {t['content'][:80]}")

## Step 5 — Load PaliGemma-3B with 4-bit QLoRA

In [ ]:
MODEL_ID = 'google/paligemma-3b-pt-224'

print('Loading processor...')
processor = PaliGemmaProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

# float16 — no quantization needed. P100 has 16 GB VRAM.
# PaliGemma-3B: ~6 GB weights + ~4 GB activations = ~10 GB total → 6 GB free.
print('Loading model in float16...')
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False

# Freeze vision encoder + projector BEFORE wrapping with PEFT
for name, param in model.named_parameters():
    if 'vision_tower' in name or 'multi_modal_projector' in name:
        param.requires_grad = False

# Required for gradient_checkpointing + LoRA — PEFT does not set this automatically
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)

# Re-freeze vision tower — get_peft_model adds LoRA to ALL matching module names
# including SigLIP layers, so we freeze them again after wrapping.
for name, param in model.named_parameters():
    if 'vision_tower' in name or 'multi_modal_projector' in name:
        param.requires_grad = False

model.print_trainable_parameters()

vram = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nVRAM used: {vram:.1f} GB / {total:.1f} GB total ({total-vram:.1f} GB free)')
print('✅ Model ready')

## Step 6 — Dataset Class
Uses PaliGemma's `suffix` parameter which automatically masks prompt tokens in labels (sets them to -100).

In [ ]:
MAX_LEN = 256

def build_prompt(conversations: list[dict]) -> tuple[str, str]:
    """
    Build (prompt, answer) from conversation turns.
    prompt  = all turns except final assistant answer
    answer  = final assistant answer (the training target)
    """
    parts  = []
    answer = ''
    for i, turn in enumerate(conversations):
        is_last = (i == len(conversations) - 1)
        if turn['role'] == 'user':
            parts.append(f"Q: {turn['content']}")
        elif not is_last:
            parts.append(f"A: {turn['content']}")
        else:
            answer = turn['content']
    return ' '.join(parts) + ' A:', answer


class VQADialogDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        prompt, answer = build_prompt(sample['conversations'])
        image  = fetch_image(sample['image_id'])

        # PaliGemma processor: `suffix` = answer text
        # It automatically creates labels with -100 on prompt tokens
        inputs = self.processor(
            images=image,
            text=prompt,
            suffix=answer,
            return_tensors='pt',
            padding='max_length',
            max_length=MAX_LEN,
            truncation=True,
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}


def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}


train_ds = VQADialogDataset(train_data, processor)
val_ds   = VQADialogDataset(val_data,   processor)

# Sanity check on one sample
s = train_ds[0]
print('Keys:', list(s.keys()))
print('input_ids shape   :', s['input_ids'].shape)
print('pixel_values shape:', s['pixel_values'].shape)
print('labels shape      :', s['labels'].shape)
n_answer_tokens = (s['labels'] != -100).sum().item()
print(f'Answer tokens in labels (non-masked): {n_answer_tokens}')
assert n_answer_tokens > 0, 'Labels are all -100! Check build_prompt()'
print('✅ Dataset class OK')

## Step 7 — Training
~2–3 hours on Kaggle P100. You can keep the tab open — Kaggle keeps running up to 12 hours.

In [ ]:
# OUTPUT_DIR is set in Step 2 (platform-aware: Kaggle vs Colab)
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='adamw_torch',
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    logging_steps=20,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print('Starting training...')
result = trainer.train()
print(f'\n✅ Done! Loss: {result.training_loss:.4f} | Time: {result.metrics["train_runtime"]/60:.1f} min')

## Step 8 — Plot Training Curves

In [ ]:
history = trainer.state.log_history

tr_steps  = [x['step'] for x in history if 'loss'      in x and 'eval_loss' not in x]
tr_losses = [x['loss'] for x in history if 'loss'      in x and 'eval_loss' not in x]
ev_steps  = [x['step'] for x in history if 'eval_loss' in x]
ev_losses = [x['eval_loss'] for x in history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tr_steps, tr_losses, label='Train Loss', color='#6366f1', linewidth=2)
ax.plot(ev_steps, ev_losses, label='Val Loss',   color='#f59e0b', linewidth=2, marker='o')
ax.set_xlabel('Steps'); ax.set_ylabel('Loss')
ax.set_title('PaliGemma-3B QLoRA — Multi-Turn Visual QA')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'✅ Saved to {SAVE_DIR}/training_curves.png')

## Step 9 — Evaluation: Zero-shot vs Fine-tuned

In [ ]:
from tqdm.auto import tqdm


def generate_answer(mdl, proc, image, prompt, max_new=40):
    # device_map='auto' shards the model — mdl.device is unreliable.
    # Use the embedding layer's device as the canonical input device.
    device = next(mdl.parameters()).device
    inputs = proc(
        images=image, text=prompt,
        return_tensors='pt', padding='longest', truncation=True, max_length=MAX_LEN,
    ).to(device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new, do_sample=False, num_beams=3)
    # Slice off the prompt tokens — avoids unreliable string-split on echoed prompt
    decoded = proc.decode(out[0][input_len:], skip_special_tokens=True).strip()
    return decoded


def exact_match(preds, refs):
    hits = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return hits / len(preds) if preds else 0.0


# Evaluate on 100 val samples
eval_subset = random.sample(val_data, min(100, len(val_data)))
ft_preds, gt_refs = [], []

model.eval()
for sample in tqdm(eval_subset, desc='Evaluating fine-tuned'):
    prompt, answer = build_prompt(sample['conversations'])
    image  = fetch_image(sample['image_id'])
    pred   = generate_answer(model, processor, image, prompt)
    ft_preds.append(pred)
    gt_refs.append(answer)

ft_acc = exact_match(ft_preds, gt_refs)
print(f'\n✅ Fine-tuned Exact Match Accuracy: {ft_acc*100:.1f}%')
print('\nSample predictions:')
for i in range(min(5, len(ft_preds))):
    print(f'  GT: {gt_refs[i][:50]:50s} | Pred: {ft_preds[i][:50]}')

## Step 10 — Ablation: Conversation History Length

In [ ]:
# Only test on samples with ≥4 turns (2+ Q-A pairs)
multi_turn = [s for s in eval_subset if len(s['conversations']) >= 4]
ablation_samples = multi_turn[:50]
print(f'Multi-turn samples for ablation: {len(ablation_samples)}')

def eval_with_n_history(samples, n_turns):
    """Evaluate with only n_turns of history before the final question."""
    preds, refs = [], []
    for sample in samples:
        convs = sample['conversations']
        # Final Q-A pair
        final_q = convs[-2]['content']
        final_a = convs[-1]['content']
        # History = all prior turns, trimmed to n_turns Q-A pairs
        prior = convs[:-2]
        prior = prior[-(n_turns * 2):] if n_turns > 0 else []
        fake_sample = {'image_id': sample['image_id'],
                       'conversations': prior + [{'role':'user','content':final_q},
                                                  {'role':'assistant','content':final_a}]}
        prompt, answer = build_prompt(fake_sample['conversations'])
        image  = fetch_image(sample['image_id'])
        pred   = generate_answer(model, processor, image, prompt)
        preds.append(pred)
        refs.append(answer)
    return exact_match(preds, refs)


ablation = {}
for n in [0, 1, 2, 3]:
    acc = eval_with_n_history(ablation_samples, n)
    ablation[n] = acc
    print(f'  History {n} turn(s): {acc*100:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f'{k} turn(s)' for k in ablation],
              [v*100 for v in ablation.values()],
              color=['#e2e8f0','#a5b4fc','#6366f1','#4338ca'])
ax.set_ylabel('Exact Match Accuracy (%)')
ax.set_title('Ablation: Conversation History Length vs Accuracy')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 100); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ablation_history.png', dpi=150)
plt.show()
print(f'✅ Saved to {SAVE_DIR}/ablation_history.png')

## Step 11 — Save Model
Saves to `/kaggle/working/dlcv_paligemma/`. Download from the **Output** tab when done.

In [ ]:
ADAPTER_PATH = f'{SAVE_DIR}/lora_adapters'
model.save_pretrained(ADAPTER_PATH)
processor.save_pretrained(ADAPTER_PATH)

_ft_acc   = locals().get('ft_acc',   None)
_ablation = locals().get('ablation', None)

results = {
    'model'              : MODEL_ID,
    'lora_rank'          : 16,
    'train_samples'      : len(train_data),
    'val_samples'        : len(val_data),
    'epochs'             : 3,
    'fine_tuned_accuracy': round(_ft_acc, 4) if _ft_acc is not None else 'not_evaluated',
    'ablation_history'   : {str(k): round(v, 4) for k, v in _ablation.items()}
                           if _ablation is not None else 'not_evaluated',
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ LoRA adapters → {ADAPTER_PATH}')
print(f'✅ results.json  → {SAVE_DIR}/results.json')
print('\nOutput files:')
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = f'{SAVE_DIR}/{fname}'
    if os.path.isdir(fpath):
        n = len(os.listdir(fpath))
        print(f'  {fname}/  ({n} files)')
    else:
        print(f'  {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
print('\n⬇️  Download: Kaggle Output tab → dlcv_paligemma/')

## Step 12 — Interactive Multi-Turn Demo

In [ ]:
from IPython.display import display

def multi_turn_chat(image_source, questions):
    """
    Run multi-turn visual QA with the fine-tuned model.
    image_source: URL string or PIL Image
    questions: list of question strings
    """
    if isinstance(image_source, str):
        img = Image.open(BytesIO(requests.get(image_source, timeout=8).content)).convert('RGB')
    else:
        img = image_source

    display(img.resize((300, 300)))
    print('─' * 50)

    history = []  # list of (question, answer) tuples
    model.eval()
    for question in questions:
        # Build prompt with full history
        parts = []
        for prev_q, prev_a in history:
            parts.append(f'Q: {prev_q} A: {prev_a}')
        parts.append(f'Q: {question} A:')
        prompt = ' '.join(parts)

        answer = generate_answer(model, processor, img, prompt)
        history.append((question, answer))
        print(f'Q: {question}')
        print(f'A: {answer}')
        print('─' * 50)


# Try it out — replace with any image URL
multi_turn_chat(
    'http://images.cocodataset.org/train2017/000000000009.jpg',
    [
        'What objects are in this image?',
        'What color are they?',
        'How many objects do you see?',
        'Where are they located in the scene?',
    ]
)